In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

1. График со всеми данными
2. Графики для каждого step
3. Графики каждого импульса + 10 точек до импулься + 288 секунд релаксации
Данные в формате - номер ячейки, температура, название импульса, начальное напряжение, конечное (после релаксации)
Первый импульс - 3, последний - 57
72, 144, 288
5. Сохраняем
6. Графики каждого 10 точек до импульса + drive cycle
Данные в формате - номер ячейки, температура, название импульса (UDDS, NEDC, WLTC), начальное напряжение, конечное (после релаксации)
Идут подряд профили UDDS: 67, 69, 71
NEDC: 77, 79, 81
WLTC: 87, 89, 91


8. Сохраняем
9. Кривые заряда и разряд
61 - разряд, 63 - заряд
   
11. Сохраняем

In [2]:
def data_visualization(df, title):
    fig, (ax_u_all, ax_i_all) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    t = df['t,s']
    U = df['U,V']
    I = df['I,A']
    
    ax_u_all.plot(t, U, color='blue')
    ax_u_all.set_ylabel("U, V")
    ax_u_all.grid(True)
    
    ax_i_all.plot(t, I, color='red')
    ax_i_all.set_ylabel("I, A")
    ax_i_all.set_xlabel("t, s")
    ax_i_all.grid(True)
    fig.suptitle(title, fontsize=12)
    
    plt.tight_layout()

In [3]:
def split_to_impulses(
    df: pd.DataFrame,
    bat_num: int,
    temp: str,
    out_dir: Path
):
    IMPULSE_DURATIONS = [72, 144, 288]
    records = []
    
    out_dir.mkdir(parents=True, exist_ok=True)

    steps = sorted(df['Step'].unique())
    impulse_steps = [s for s in steps if s >= 5 and s % 2 == 1]

    EXPECTED_IMPULSES = 27
    impulse_steps = impulse_steps[:EXPECTED_IMPULSES]

    for idx, step in enumerate(impulse_steps):
        prev_step = step - 1
        next_step = step + 1

        # длительность импульса по циклу
        impulse_time = IMPULSE_DURATIONS[idx % 3]

        df_prev = df[df['Step'] == prev_step].tail(10)
        df_imp = df[df['Step'] == step]
        df_next = df[df['Step'] == next_step]

        # начальное напряжение
        U_start = df_prev['U,V'].mean()

        # конечное напряжение — последние 10 точек ДО обрезки
        U_end = df_next.tail(10)['U,V'].mean()

        # 288 секунд релаксации
        t0 = df_next['t,s'].iloc[0]
        df_relax = df_next[df_next['t,s'] <= t0 + 288]

        df_out = pd.concat([df_imp, df_relax])

        t_start = df_out['t,s'].iloc[0]
        df_out = df_out.copy()
        df_out['t,s'] = df_out['t,s'] - t_start

        filename = f"{bat_num}_{temp}_impulse_{impulse_time}_s_{idx+1:02d}.csv"
        df_out.to_csv(out_dir / filename, index=False)

        impulse_type = f"impulse_{impulse_time}_s"

        records.append({
            "bat_num": bat_num,
            "temp": temp,
            "profile": impulse_type,
            "U_start_V": U_start,
            "U_end_V": U_end,
            "file_name": filename
        })
    return records

        

In [4]:
def split_drive_cycles(
    df: pd.DataFrame,
    bat_num: int,
    temp: str,
    out_dir: Path
):

    out_dir.mkdir(parents=True, exist_ok=True)
    records = []

    # Определяем все шаги
    steps = sorted(df['Step'].unique())

    # Задаём маппинг профилей на шаги
    profile_map = {
        'UDDS': [67, 69, 71],
        'NEDC': [77, 79, 81],
        'WLTC': [87, 89, 91]
    }

    for cycle_name, cycle_steps in profile_map.items():
        for idx, step in enumerate(cycle_steps):
            prev_step = step - 1  # 10 точек до цикла
            next_step = step + 1  # релаксация после цикла

            df_prev = df[df['Step'] == prev_step].tail(10)
            df_cycle = df[df['Step'] == step]
            df_next_full = df[df['Step'] == next_step]

            # начальное и конечное напряжение
            U_start = df_prev['U,V'].mean()
            U_end = df_next_full.tail(10)['U,V'].mean()

            # формируем полный DataFrame
            df_out = df_cycle

            # обнуляем время
            t_start = df_out['t,s'].iloc[0]
            df_out = df_out.copy()
            df_out['t,s'] = df_out['t,s'] - t_start

            # имя файла
            filename = f"{bat_num}_{temp}_{cycle_name}_{idx+1:02d}.csv"
            df_out.to_csv(out_dir / filename, index=False)

            records.append({
                "bat_num": bat_num,
                "temp": temp,
                "profile": cycle_name,
                "U_start_V": U_start,
                "U_end_V": U_end,
                "file_name": filename
            })

    return records


In [5]:
def split_charge_discharge(
    df: pd.DataFrame,
    bat_num: int,
    temp: str,
    out_dir: Path
):
    out_dir.mkdir(parents=True, exist_ok=True)
    records = []

    # mapping: Step -> профиль
    profile_map = {
        61: "discharge",
        63: "charge"
    }

    for step, profile_name in profile_map.items():
        df_step = df[df['Step'] == step]

        # обнуляем время
        t_start = df_step['t,s'].iloc[0]
        df_out = df_step.copy()
        df_out['t,s'] = df_out['t,s'] - t_start

        # сохраняем CSV
        file_name = f"{bat_num}_{temp}_{profile_name}.csv"
        file_path = out_dir / file_name
        df_out.to_csv(file_path, index=False)

        # сохраняем метаданные
        records.append({
            "bat_num": bat_num,
            "temp": temp,
            "profile": profile_name,
            "file_name": file_name
        })

    return records


In [6]:
current_dir = Path.cwd()
print(f"Current dir: {current_dir}")

project_root = current_dir.parent 
raw_data_dir = project_root / "raw_data"
T = ["+25", "+30", "+35"]
start_num = 1
end_num = 6 

# for temp in T:
#     for i in range(start_num, end_num):
#         csv_file = raw_data_dir / temp / str(i) / f"{i} {temp}.csv"

#         df = pd.read_csv(csv_file)

#         data_visualization(df, f"{i} {temp}.csv")
        

Current dir: D:\Documents\battery\Parametrization\USBEREIT conference\Data_processing\Data_preprocessing\code


In [7]:
# for temp in T:
#     for i in range(start_num, end_num):
#         csv_file = raw_data_dir / temp / str(i) / f"{i} {temp}.csv"
        
#         df = pd.read_csv(csv_file)

#         unique_steps = df['Step'].unique()

#         for step in unique_steps:
#             df_step = df[df['Step'] == step]

#             data_visualization(df_step, f"{i} {temp}.csv Step: {step} {df['Step Type']}")

        

In [8]:
all_records = []
profiles_for_parametrization_root = project_root / "profiles"

for temp in T:
    for i in range(start_num, end_num):
        csv_file = raw_data_dir / temp / str(i) / f"{i} {temp}.csv"
        df = pd.read_csv(csv_file)

        records_imp = split_to_impulses(
            df=df,
            bat_num=i,
            temp=temp,
            out_dir=profiles_for_parametrization_root
        )

        records_dc = split_drive_cycles(
            df=df,
            bat_num=i,
            temp=temp,
            out_dir=profiles_for_parametrization_root
        )

        
        all_records.extend(records_imp)

        all_records.extend(records_dc)

        




summary_df = pd.DataFrame(all_records)
summary_df.to_csv(profiles_for_parametrization_root / "profiles_description.csv", index=False)



In [10]:
all_charge_discharge = []

for temp in T:
    for i in range(start_num, end_num):
        csv_file = raw_data_dir / temp / str(i) / f"{i} {temp}.csv"
        df = pd.read_csv(csv_file)

        records_cd = split_charge_discharge(
            df=df,
            bat_num=i,
            temp=temp,
            out_dir=project_root / "low_current_charge_and_discharge" 
        )
        all_charge_discharge.extend(records_cd)

# сохраняем сводный CSV
summary_cd = pd.DataFrame(all_charge_discharge)
summary_cd.to_csv(project_root / "low_current_charge_and_discharge" / "charge_discharge_description.csv", index=False)
